# Phase 3: Reasoning Finetuning & Evaluation — Nepali (Model L)
### Course: Language Models and Agents (Monsoon 2026)
**Assignment:** Individual Project — Building & Analyzing Monolingual Transformer LMs in Two Indian Languages  
**Language:** Nepali (`ne`) — Low Resource | **Branch:** `phase-3`

This notebook finetunes the pretrained Nepali Model L (25.33M parameters) on 7,000 synthetic comparative and transitive reasoning samples, evaluates exact match accuracy on held-out test data, and computes post-finetune attention shifts.

In [ ]:
# CELL 1 — GPU VERIFICATION & ENVIRONMENT SETUP
!pip install -q sentencepiece pyarrow pandas matplotlib seaborn numpy tqdm gdown

import os
import sys
import torch

print("CUDA Available:", torch.cuda.is_available())
if torch.cuda.is_available():
    print("GPU Device   :", torch.cuda.get_device_name(0))
    print(f"GPU Memory   : {torch.cuda.get_device_properties(0).total_memory / (1024**3):.2f} GB")
else:
    print("Running on CPU (GPU recommended for fast finetuning)")

In [ ]:
# CELL 2 — AUTOMATIC PROJECT & DATA PATH SETUP
import os
import shutil
from pathlib import Path

OUTPUT_DIR = "/kaggle/working/nepali_finetuned_checkpoints"
os.makedirs(OUTPUT_DIR, exist_ok=True)

# Locate project files automatically across /kaggle/input and current workspace
CODE_ROOT = None
DATA_DIR = None
PRETRAINED_CKPT = None
TOKENIZER_MODEL = None
CONFIG_FILE = None

for root, dirs, files in os.walk("/kaggle/input"):
    if "transformer.py" in files and "nepali" in root:
        CODE_ROOT = str(Path(root).parents[1])
    if "train.parquet" in files and "reasoning" in root and "nepali" in root:
        DATA_DIR = root
    if "checkpoint_best.pt" in files and "nepali" in root:
        PRETRAINED_CKPT = os.path.join(root, "checkpoint_best.pt")
    if "tokenizer.model" in files and "nepali" in root:
        TOKENIZER_MODEL = os.path.join(root, "tokenizer.model")
    if "model_config.json" in files and "nepali" in root:
        CONFIG_FILE = os.path.join(root, "model_config.json")

# Fallback to local repo paths if running in repo
if CODE_ROOT is None:
    CODE_ROOT = os.getcwd()
if DATA_DIR is None:
    DATA_DIR = os.path.join(CODE_ROOT, "nepali", "data", "reasoning")
if PRETRAINED_CKPT is None:
    local_ckpt = os.path.join(CODE_ROOT, "nepali", "checkpoints", "checkpoint_best.pt")
    if os.path.exists(local_ckpt):
        PRETRAINED_CKPT = local_ckpt
    else:
        # Auto-download from Drive if missing
        print("Downloading Nepali pretrained checkpoint from Google Drive...")
        os.makedirs(os.path.join(CODE_ROOT, "nepali", "checkpoints"), exist_ok=True)
        !gdown --id 1ocqIcYfPMbqKulJKXM90mGxmSBglzlmU -O {local_ckpt}
        PRETRAINED_CKPT = local_ckpt
if TOKENIZER_MODEL is None:
    TOKENIZER_MODEL = os.path.join(CODE_ROOT, "nepali", "tokenizer", "unigram_10k", "tokenizer.model")
if CONFIG_FILE is None:
    CONFIG_FILE = os.path.join(CODE_ROOT, "nepali", "configs", "model_config.json")

TRAIN_PARQUET = os.path.join(DATA_DIR, "train.parquet")
VAL_PARQUET = os.path.join(DATA_DIR, "val.parquet")
TEST_PARQUET = os.path.join(DATA_DIR, "test.parquet")

print("=" * 60)
print("NEPALI PHASE 3 SETUP VERIFICATION")
print("=" * 60)
print("Code Root       :", CODE_ROOT)
print("Pretrained Ckpt :", PRETRAINED_CKPT)
print("Tokenizer Model :", TOKENIZER_MODEL)
print("Model Config    :", CONFIG_FILE)
print("Train Parquet   :", TRAIN_PARQUET)
print("Val Parquet     :", VAL_PARQUET)
print("Test Parquet    :", TEST_PARQUET)
print("=" * 60)

In [ ]:
# CELL 3 — RUN SUPERVISED REASONING FINETUNING (5 EPOCHS)
finetune_script = os.path.join(CODE_ROOT, "scripts", "finetune_reasoning.py")

!python {finetune_script} \
    --language Nepali \
    --pretrained_checkpoint {PRETRAINED_CKPT} \
    --train_parquet {TRAIN_PARQUET} \
    --val_parquet {VAL_PARQUET} \
    --tokenizer_model {TOKENIZER_MODEL} \
    --config_file {CONFIG_FILE} \
    --output_dir /kaggle/working/nepali_finetuned_checkpoints \
    --batch_size 32 \
    --epochs 5 \
    --learning_rate 2e-4 \
    --weight_decay 0.01 \
    --warmup_ratio 0.05 \
    --use_amp \
    --device cuda

In [ ]:
# CELL 4 — EVALUATE REASONING EXACT MATCH (EM) ACCURACY
eval_script = os.path.join(CODE_ROOT, "scripts", "evaluate_reasoning.py")
best_ft_ckpt = "/kaggle/working/nepali_finetuned_checkpoints/checkpoints/checkpoint_best.pt"
eval_output = "/kaggle/working/nepali_reasoning_eval.json"

!python {eval_script} \
    --config_file {CONFIG_FILE} \
    --checkpoint {best_ft_ckpt} \
    --test_parquet {TEST_PARQUET} \
    --tokenizer_model {TOKENIZER_MODEL} \
    --output_file {eval_output} \
    --device cuda

In [ ]:
# CELL 5 — POST-FINETUNING ATTENTION SHIFT ANALYSIS
attn_script = os.path.join(CODE_ROOT, "scripts", "compare_finetuned_attention.py")
best_ft_ckpt = "/kaggle/working/nepali_finetuned_checkpoints/checkpoints/checkpoint_best.pt"
attn_figure = "/kaggle/working/nepali_attention_comparison.png"

!python {attn_script} \
    --language Nepali \
    --config_file {CONFIG_FILE} \
    --pretrained_checkpoint {PRETRAINED_CKPT} \
    --finetuned_checkpoint {best_ft_ckpt} \
    --tokenizer_model {TOKENIZER_MODEL} \
    --output_figure {attn_figure} \
    --device cuda

from IPython.display import Image, display
if os.path.exists(attn_figure):
    display(Image(attn_figure))